# Detecção e Segmentação de Objetos em Operações Logísticas com YOLO
# Nome: Thais Silva Ferreira
## Contexto do problema

Um centro de distribuição deseja utilizar visão computacional para apoiar a identificação e localização de pacotes em esteiras, áreas de triagem e operações de armazenagem.

Nesta atividade, será utilizado o **Package Segmentation Dataset** da Ultralytics para realizar **detecção e segmentação de instâncias** usando YOLO.

Fluxo da prática:

**Dataset anotado → Modelo pré-treinado → Fine-tuning → Inferência → Avaliação → Análise das predições**


## Objetivos

- utilizar um modelo YOLO pré-treinado;
- realizar fine-tuning em um domínio específico;
- executar inferências;
- acessar classes, confidence scores, bounding boxes e máscaras;
- avaliar o modelo com Precision, Recall e mAP;
- investigar o efeito de confidence threshold e IoU;
- comparar detecção e segmentação em uma aplicação real.


In [1]:
!pip install -q -U ultralytics


In [2]:
from ultralytics import YOLO
import ultralytics
import torch
import glob
import random
import matplotlib.pyplot as plt


Creating new Ultralytics Settings v0.0.7 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Administrator\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [3]:
print("Versão do Ultralytics:", ultralytics.__version__)
print("Versão do PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Executando em CPU")


Versão do Ultralytics: 8.4.129
Versão do PyTorch: 2.13.0+cpu
CUDA disponível: False
Executando em CPU


## Dataset

Será utilizado o **Package Segmentation Dataset** disponibilizado pelo Ultralytics.

Características:

- 2.197 imagens;
- 1.920 imagens de treinamento;
- 188 imagens de validação;
- 89 imagens de teste;
- uma classe: `package`;
- bounding boxes e máscaras de segmentação.

O dataset é baixado automaticamente pelo Ultralytics ao utilizar `package-seg.yaml`.


In [4]:
# Modelo YOLO pré-treinado para segmentação
model = YOLO("yolo26n-seg.pt")


In [5]:
# Fine-tuning
results = model.train(
    data="package-seg.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project="package_segmentation",
    name="yolo_logistics"
)


Ultralytics 8.4.129  Python-3.11.9 torch-2.13.0+cpu CPU (Intel Core i5-7200U 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=package-seg.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_logistics, nbs=64, nms=False, opse

KeyboardInterrupt: 

## Carregamento do melhor modelo

Após o treinamento, o arquivo `best.pt` contém o modelo com os melhores resultados obtidos durante o fine-tuning.


In [ ]:
best_model_path = "package_segmentation/yolo_logistics/weights/best.pt"
model = YOLO(best_model_path)

print("Modelo carregado com sucesso!")
print("Caminho:", best_model_path)


## Avaliação no conjunto de validação

Serão utilizadas as métricas:

- Precision;
- Recall;
- mAP@50;
- mAP@50-95.

As métricas serão analisadas tanto para as bounding boxes quanto para as máscaras de segmentação.


In [ ]:
metrics = model.val(
    data="package-seg.yaml",
    split="val"
)

print("Avaliação concluída.")


In [ ]:
print("===== DETECÇÃO =====")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

print("\n===== SEGMENTAÇÃO =====")
print(f"Precision: {metrics.seg.mp:.4f}")
print(f"Recall: {metrics.seg.mr:.4f}")
print(f"mAP@50: {metrics.seg.map50:.4f}")
print(f"mAP@50-95: {metrics.seg.map:.4f}")


## Localização das imagens de teste


In [ ]:
# Localiza automaticamente as imagens de teste após o download do dataset
test_images = glob.glob(
    "/content/**/package-seg/images/test/*",
    recursive=True
)

print("Quantidade de imagens encontradas:", len(test_images))

if len(test_images) == 0:
    print("Execute o treinamento ou uma validação primeiro para que o dataset seja baixado.")


In [ ]:
image_path = random.choice(test_images)

print("Imagem selecionada:")
print(image_path)


## Inferência


In [ ]:
results = model.predict(
    source=image_path,
    conf=0.25,
    iou=0.45,
    verbose=False
)

result = results[0]


In [ ]:
annotated_image = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title("Detecção e Segmentação de Pacotes")
plt.show()


## Classes, confidence scores e bounding boxes


In [ ]:
if result.boxes is not None and len(result.boxes) > 0:

    classes = result.boxes.cls.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()

    for i in range(len(classes)):
        class_id = int(classes[i])

        print(f"\nObjeto {i + 1}")
        print(f"Classe: {result.names[class_id]}")
        print(f"Confidence: {confidences[i]:.4f}")
        print(f"Bounding Box [x1, y1, x2, y2]: {boxes[i]}")

else:
    print("Nenhum objeto detectado.")


## Máscaras de segmentação


In [ ]:
if result.masks is not None:
    masks = result.masks.data.cpu().numpy()

    print("Quantidade de máscaras:", len(masks))
    print("Formato das máscaras:", masks.shape)
else:
    print("Nenhuma máscara encontrada.")


In [ ]:
if result.masks is not None:

    masks = result.masks.data.cpu().numpy()
    quantidade = min(len(masks), 5)

    plt.figure(figsize=(15, 5))

    for i in range(quantidade):
        plt.subplot(1, quantidade, i + 1)
        plt.imshow(masks[i], cmap="gray")
        plt.title(f"Máscara {i + 1}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## Análise do Confidence Threshold

O confidence threshold define a confiança mínima para aceitar uma predição.

Valores menores tendem a gerar mais detecções, podendo aumentar falsos positivos. Valores maiores tornam o modelo mais rigoroso, mas podem deixar de detectar alguns pacotes.


In [ ]:
confidence_values = [0.10, 0.25, 0.50, 0.75]

for conf in confidence_values:

    results = model.predict(
        source=image_path,
        conf=conf,
        iou=0.45,
        verbose=False
    )

    prediction = results[0]
    annotated_image = prediction.plot()

    quantidade = len(prediction.boxes) if prediction.boxes is not None else 0

    plt.figure(figsize=(10, 6))
    plt.imshow(annotated_image)
    plt.axis("off")
    plt.title(f"Confidence = {conf} | Objetos detectados = {quantidade}")
    plt.show()


## Análise do IoU

O parâmetro IoU influencia a supressão de detecções sobrepostas. Nesta etapa, serão comparados diferentes valores para observar seu impacto sobre o resultado final.


In [ ]:
iou_values = [0.30, 0.45, 0.60, 0.75]

for iou in iou_values:

    results = model.predict(
        source=image_path,
        conf=0.25,
        iou=iou,
        verbose=False
    )

    prediction = results[0]
    annotated_image = prediction.plot()

    quantidade = len(prediction.boxes) if prediction.boxes is not None else 0

    plt.figure(figsize=(10, 6))
    plt.imshow(annotated_image)
    plt.axis("off")
    plt.title(f"IoU = {iou} | Objetos detectados = {quantidade}")
    plt.show()


## Avaliação final no conjunto de teste


In [ ]:
metrics_test = model.val(
    data="package-seg.yaml",
    split="test"
)


In [ ]:
print("========== DETECÇÃO ==========")
print(f"Precision: {metrics_test.box.mp:.4f}")
print(f"Recall: {metrics_test.box.mr:.4f}")
print(f"mAP@50: {metrics_test.box.map50:.4f}")
print(f"mAP@50-95: {metrics_test.box.map:.4f}")

print("\n========== SEGMENTAÇÃO ==========")
print(f"Precision: {metrics_test.seg.mp:.4f}")
print(f"Recall: {metrics_test.seg.mr:.4f}")
print(f"mAP@50: {metrics_test.seg.map50:.4f}")
print(f"mAP@50-95: {metrics_test.seg.map:.4f}")


## Comparação entre Detecção e Segmentação

### Detecção de objetos

A detecção identifica a localização aproximada dos pacotes utilizando bounding boxes. É adequada para:

- contagem de pacotes;
- localização aproximada;
- monitoramento de esteiras;
- identificação da presença de objetos.

### Segmentação de instâncias

A segmentação identifica os pixels pertencentes a cada pacote individualmente. É adequada para:

- manipulação robótica;
- identificação precisa dos limites;
- separação de objetos próximos;
- análise de objetos sobrepostos;
- cálculo da área ocupada.

### Aplicação logística

Para tarefas simples de contagem, a detecção pode ser suficiente. Para robôs que precisam manipular pacotes ou cenários com objetos muito próximos, a segmentação oferece informações espaciais mais precisas.


# Conclusão

Nesta atividade, foi desenvolvido um sistema de visão computacional para detecção e segmentação de pacotes em operações logísticas utilizando YOLO.

Foi utilizado um modelo pré-treinado e realizado fine-tuning com o Package Segmentation Dataset. O modelo foi utilizado para identificar pacotes e acessar suas classes, confidence scores, bounding boxes e máscaras de segmentação.

A avaliação foi realizada utilizando Precision, Recall e mAP. Também foram analisados os efeitos da alteração do confidence threshold e do IoU.

A detecção é adequada para localização e contagem de pacotes. A segmentação fornece uma representação mais precisa dos objetos, sendo especialmente útil em robótica, automação de armazéns e manipulação de pacotes próximos ou sobrepostos.
